# Loading your own data

This notebook builds an `InteractionDataset` from a table you already have, trains FedNCF, and reports full-ranking NDCG@10 and HR@10 on the test split.

The first table is synthetic: users, items, ratings and timestamps drawn in memory. The second is [HetRec 2011 Last.fm 2K](https://grouplens.org/datasets/hetrec-2011/), which you download and unzip into the working directory. Both go through `load_dataframe_dataset`.

FedNCF (Perifanis and Efraimidis, *Knowledge-Based Systems*, 2022) is neural collaborative filtering with one user per client. The server holds the item embeddings and the shared network. Each client holds that user's embeddings. The wiring below matches the MovieLens quickstart (`examples/quickstart.ipynb`): the same split ratios, the same aggregators, validation during training, and a test score from the best validation round. The model is smaller and the run is a few rounds, so both tables finish on a CPU. The quickstart is the 100-round MovieLens configuration.

## Install

FedRecLab is installed from GitHub. The command also installs NumPy, PyTorch, pandas, tqdm and Cornac. PyTorch is a large download. Run this cell before any import. The install itself does not depend on the repository working directory, so the same cell works in Colab from a fresh environment. The Last.fm section later reads `user_artists.dat` from that working directory.

In [ ]:
%pip install "git+https://github.com/aymenkhs/FedRecLab.git"

## What the table contains

`load_dataframe_dataset` reads a pandas DataFrame and returns an `InteractionDataset`. That object is the only copy of the interactions. It can filter rows, then remaps user and item ids to contiguous integers starting at 0, splits, and assigns clients.

The frame needs `user_id` and `item_id`. `rating` and `timestamp` are optional. A chronological split reads `timestamp`. Ids may be any sortable values; the synthetic users below start at 1001. After remapping, 0 is a real user and a real item. `user_id_map` and `item_id_map` translate original ids to the new ones.

The loader accepts other header names. FedNCF and the evaluator read columns named `user_id`, `item_id`, and `rating` when that column is present, so this notebook renames foreign headers to those names before loading.

Rows are counted as they stand. A repeated `(user, item)` pair counts more than once toward the frequency filters. Drop duplicate pairs in the frame when each pair should count once.

The split is stratified by user, 10% validation and 20% test, as in the quickstart. Cornac turns each fraction into a count with `ceil`, separately for every user. `ceil(0.1 * n) + ceil(0.2 * n)` is 2 when a user has one or two rows, and the split raises because the holdout does not fit. Every synthetic user has 12 rows, so that table needs no frequency cut. Last.fm sets `min_user_interactions=3`.

## A synthetic interaction table

Draw 40 users and 60 items with a seeded NumPy generator. Each user has 12 distinct items. Ratings are integers from 1 to 5. Timestamps increase within each user, which gives the chronological split an order to follow. User ids start at 1001 and item ids at 50001 so the remapping is visible in the next step.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

n_users = 40
n_items = 60
per_user = 12

user_ids = np.repeat(np.arange(1001, 1001 + n_users), per_user)
item_pool = np.arange(50_001, 50_001 + n_items)
item_ids = np.concatenate(
    [rng.choice(item_pool, size=per_user, replace=False) for _ in range(n_users)]
)
ratings = rng.integers(1, 6, size=user_ids.shape[0]).astype(float)
base = rng.integers(1_600_000_000, 1_700_000_000, size=n_users)
offsets = np.sort(rng.integers(0, 10_000_000, size=(n_users, per_user)), axis=1)
timestamps = (base[:, None] + offsets).ravel()

interactions = pd.DataFrame(
    {
        "user_id": user_ids,
        "item_id": item_ids,
        "rating": ratings,
        "timestamp": timestamps,
    }
)
print(
    f"{interactions['user_id'].nunique()} users, "
    f"{interactions['item_id'].nunique()} items, {len(interactions)} rows"
)
interactions.head()

## Build the dataset

`StratifiedSplitStrategy(..., chrono=True)` keeps each user's earlier interactions for training and holds out the later ones: 10% validation, 20% test. Training reads the training split. The evaluator reads validation and test.

`CrossDevicePartitionStrategy` assigns each training user to their own client, so the number of clients is the number of training users.

`include_validation` and `include_test` on the loader match the strategy. With either flag left false, the loader drops that held-out index. `k_core=0`, `min_user_interactions=0` and `min_item_interactions=0` leave this table unfiltered. `seed=42` is stored on the dataset. The split strategy carries its own seed.

In [ ]:
from fedreclab.data.client_partitioning import CrossDevicePartitionStrategy
from fedreclab.data.load_data import load_dataframe_dataset
from fedreclab.data.split import StratifiedSplitStrategy

dataset = load_dataframe_dataset(
    interactions,
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True,
        include_test=True,
        val_size=0.1,
        test_size=0.2,
        chrono=True,
        by="user",
        seed=SEED,
    ),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True,
    include_test=True,
    seed=SEED,
    k_core=0,
    min_user_interactions=0,
    min_item_interactions=0,
)

print(
    f"{dataset.num_users} users, {dataset.num_items} items, "
    f"{dataset.num_interactions} interactions, {dataset.num_clients} clients"
)
print("original user 1001 ->", int(dataset.user_id_map[1001]))
print("original item 50001 ->", int(dataset.item_id_map[50001]))
print(dataset.interaction_data["split"].value_counts())
dataset.interaction_data.head()

## Train and evaluate

The function below builds every object the server owns, trains, and prints the test metrics. Call it once per dataset. Client algorithms, selectors, aggregators and evaluators are single-use: a second server needs new ones. The dataset may be shared. Each call builds new components.

`FedNCFClientAlgorithm` runs local training. One algorithm object is shared across clients. Each client's user embeddings and optimiser state live in a client state that the server creates. `predictive_factor` is the latent width and must be the same value on the client and the server. This demo uses 8. The quickstart uses 32. Observed interactions are implicit positives, four unobserved items are sampled as negatives, and the loss is binary cross-entropy. `local_epochs=1` keeps the demo short. The quickstart uses 5. A rating stored on the row is not the training target under this loss.

`FullRankingEvaluator` scores, for each user, every catalogue item except items from the excluded splits, then averages HR and NDCG at 10. Validation candidates drop that user's training items. Test candidates drop training and validation items. Scoring uses the client algorithm passed in, so the evaluator and the server share that algorithm.

`RandomClientSelection` draws `client_fraction` of the clients each round. The count is `int(num_clients * fraction)`, so 2% of 40 clients is 0 and the selector raises. The synthetic run therefore draws a quarter of the clients. Last.fm draws 2%, the quickstart fraction. `seed=42` on the server seeds PyTorch and, because the selector is built without its own seed, the client draws as well.

The uploaded groups are aggregated separately:

- `item_embeddings` uses `SimpleAvg`. Clients upload the item rows they trained on, and each row is the uniform mean of the clients that sent it.
- `shared_layers` and `output_weights` use `FedAvg`, a mean weighted by how many training interactions the client has.

Item rows are uploaded in the clear, so the server sees which items each selected client updated. The paper's secure-aggregation masking is not part of this run.

`early_stopping` follows validation `ndcg@10`. Patience is 5 checks. Three rounds finish before that patience is spent. The test score uses the snapshot from the best validation round (`best_global_parameters`, `best_client_states`, `best_extra_payload`), not the parameters of the final round.

On Last.fm each validation scores every remaining artist for every user. Three rounds plus the test pass take a few minutes on CPU.

In [ ]:
from fedreclab.data.evaluator import FullRankingEvaluator
from fedreclab.federated.aggregation import FedAvg, SimpleAvg
from fedreclab.federated.client_selection import RandomClientSelection
from fedreclab.methods.fedncf.fedncf import FedNCF
from fedreclab.methods.fedncf.fedncf_client import FedNCFClientAlgorithm

PREDICTIVE_FACTOR = 8


def train_and_evaluate(interaction_dataset, num_rounds, client_fraction):
    """Train a new FedNCF server and print test NDCG@10 and HR@10.

    Builds a new client algorithm, selector, aggregators and evaluator.
    Those objects are single-use, so each dataset needs its own call.
    """
    client = FedNCFClientAlgorithm(
        predictive_factor=PREDICTIVE_FACTOR,
        local_epochs=1,
        lr=5e-4,
        batch_size=128,
        neg_samples=4,
    )
    evaluator = FullRankingEvaluator(
        metrics=["ndcg", "hr"],
        topk=10,
        interaction_dataset=interaction_dataset,
        client_algorithm=client,
    )
    server = FedNCF(
        predictive_factor=PREDICTIVE_FACTOR,
        interaction_dataset=interaction_dataset,
        client_algorithm=client,
        num_rounds=num_rounds,
        seed=SEED,
        client_selection_strategy=RandomClientSelection(
            interaction_dataset.num_clients, client_fraction, "frac"
        ),
        aggregation_strategies={
            "item_embeddings": SimpleAvg(num_clients=interaction_dataset.num_clients),
            "shared_layers": FedAvg(num_clients=interaction_dataset.num_clients),
            "output_weights": FedAvg(num_clients=interaction_dataset.num_clients),
        },
        evaluator=evaluator,
        early_stopping=True,
        early_stopping_parameters={
            "metric": "ndcg@10",
            "direction": "max",
            "patience": 5,
        },
    )
    server.run()
    print(f"best validation round: {server.early_stopping_best_round}")
    print(evaluator.evaluate(
        current_round=server.early_stopping_best_round,
        global_parameters=server.best_global_parameters,
        client_states=server.best_client_states,
        extra_payload=server.best_extra_payload,
        split="test",
    ))

Three rounds, with a quarter of the 40 clients selected each round. The catalogue is 60 items, so validation is quick.

In [ ]:
train_and_evaluate(dataset, num_rounds=3, client_fraction=0.25)

## Last.fm 2K

HetRec 2011 Last.fm 2K is listening counts for 1,892 users and 17,632 artists (Cantador, Brusilovsky, and Kuflik, 2011). Download the archive and unzip it so `user_artists.dat` sits in the repository root — the working directory if you started Jupyter there, or `/content` on Colab:

https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip

The GroupLens page, with the file descriptions, is https://grouplens.org/datasets/hetrec-2011/.

The zip has no top-level folder. Unzipping it writes `user_artists.dat` directly into the directory you chose. The file is tab-separated and has a header: `userID`, `artistID`, `weight`. `weight` is how many times that user listened to that artist. These rows have no timestamp. The other files in the archive (artists, tags, friends) are left unused.

Rename the headers to `user_id`, `item_id`, and `rating` before loading. FedNCF's default loss treats every listened artist as a positive. The listening count stays in `rating` and is not the target.

`min_user_interactions=3` drops users whose history cannot hold a 10% validation split and a 20% test split under Cornac's per-user `ceil`. Nine users in this file have fewer than three artists. `chrono` stays off, and `time_col=None`, because there is no timestamp column. The cell also looks one directory up, so a kernel started inside `examples/` still finds `user_artists.dat` in the repository root.

In [ ]:
from pathlib import Path

candidates = [Path("user_artists.dat"), Path("..") / "user_artists.dat"]
interactions_path = next((path for path in candidates if path.is_file()), None)
if interactions_path is None:
    raise FileNotFoundError(
        "user_artists.dat was not found in the working directory "
        f"({Path.cwd()}) or its parent. Download "
        "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip "
        "and unzip it so user_artists.dat sits in the repository root."
    )

lastfm = pd.read_csv(interactions_path, sep="\t").rename(
    columns={"userID": "user_id", "artistID": "item_id", "weight": "rating"}
)

lastfm_dataset = load_dataframe_dataset(
    lastfm,
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True,
        include_test=True,
        val_size=0.1,
        test_size=0.2,
        chrono=False,
        by="user",
        seed=SEED,
    ),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True,
    include_test=True,
    seed=SEED,
    time_col=None,
    k_core=0,
    min_user_interactions=3,
    min_item_interactions=0,
)

print(
    f"{lastfm_dataset.num_users} users, {lastfm_dataset.num_items} items, "
    f"{lastfm_dataset.num_interactions} interactions, {lastfm_dataset.num_clients} clients"
)
print("original user 2 ->", int(lastfm_dataset.user_id_map[2]))
print(lastfm_dataset.interaction_data["split"].value_counts())

Three rounds, with 2% of clients each round. This call builds a new client, selector, aggregators and evaluator. The objects from the synthetic run stay bound to that server.

In [ ]:
train_and_evaluate(lastfm_dataset, num_rounds=3, client_fraction=0.02)